# Traffic Density Detection - System Test Notebook
Notebook này giúp bạn khởi động Backend, chạy Detection (Module A), và kiểm tra truy vấn Database, gọi API Machine Learning dự báo.

**Lưu ý**: Đảm bảo bạn đã chọn Kernel (Môi trường Python) là `venv` của dự án.

In [1]:
import os
import sys
from dotenv import load_dotenv

# Nạp các biến môi trường từ file .env
load_dotenv()

db_url = os.getenv("DB_URL")
if db_url:
    print(f"✅ Đã nạp thành công DB_URL: {db_url[:20]}... (ẩn phần sau để bảo mật)")
else:
    print("❌ Lỗi: Không tìm thấy DB_URL trong file .env.")

print(f"📁 Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"🐍 Trình thông dịch Python đang dùng: {sys.executable}")

✅ Đã nạp thành công DB_URL: mongodb+srv://nguyen... (ẩn phần sau để bảo mật)
📁 Thư mục làm việc hiện tại: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system
🐍 Trình thông dịch Python đang dùng: d:\GIT REPO\trafffic-density-analysis-system\traffic-density-analysis-system\venv\Scripts\python.exe


In [2]:
import subprocess
import time
import requests
import sys

print("🚀 Đang khởi động Backend Server (Module B) ở cổng 8000...")

# Bật Uvicorn thông qua sys.executable để đảm bảo dùng đúng môi trường ảo (venv)
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open('test_backend.log', 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(6) # Chờ 6 giây cho server khởi động

try:
    res = requests.get("http://127.0.0.1:8000/docs")
    if res.status_code == 200:
        print("✅ Backend đã hoạt động tốt tại http://127.0.0.1:8000")
except Exception as e:
    print(f"⚠️ Không thể kết nối Backend. Hãy xem file test_backend.log. Lỗi: {e}")

🚀 Đang khởi động Backend Server (Module B) ở cổng 8000...
✅ Backend đã hoạt động tốt tại http://127.0.0.1:8000


In [3]:
import subprocess
import sys

print("👁️ Bắt đầu chạy Nhận diện và Tracking (Module A)...")
print("👉 Hãy chú ý cửa sổ Video hiện lên. Bấm phím 'q' trên cửa sổ video để thoát.")

# Dùng subprocess thay cho !python để tránh lỗi do đường dẫn thư mục có dấu cách
process = subprocess.run([sys.executable, "-m", "detection.for_test.main_v9"])

print(f"✅ Đã kết thúc tiến trình nhận diện. (Mã thoát: {process.returncode})")

👁️ Bắt đầu chạy Nhận diện và Tracking (Module A)...
👉 Hãy chú ý cửa sổ Video hiện lên. Bấm phím 'q' trên cửa sổ video để thoát.
✅ Đã kết thúc tiến trình nhận diện. (Mã thoát: 0)


In [12]:
from pymongo import MongoClient
import pandas as pd

print("🔍 Đang truy vấn MongoDB (Collection: vehicle_detections)...")

client = MongoClient(db_url)
# Tên database chuẩn của hệ thống là traffic_density
db = client.get_database("traffic_density") 

count = db.vehicle_detections.count_documents({})
print(f"📊 Hệ thống đã ghi nhận: {count} sự kiện xe qua vạch.")

if count > 0:
    cursor = db.vehicle_detections.find({}, {"_id": 0}).sort("timestamp", -1).limit(3)
    df = pd.DataFrame(list(cursor))
    print("\n🚗 3 bản ghi mới nhất:")
    display(df)

🔍 Đang truy vấn MongoDB (Collection: vehicle_detections)...
📊 Hệ thống đã ghi nhận: 4770 sự kiện xe qua vạch.

🚗 3 bản ghi mới nhất:


,event_id,camera_id,track_id,vehicle_type,density,direction,event_type,confidence,timestamp
0,c23761fc-ec09-4d5f-9fb1-204135dd310e,CAM_01,1,car,HIGH,inbound,zone_entry,0.9435,2026-05-02 15:40:55.465
1,613393dd-3731-4fd4-8d8f-98234dc23cb4,CAM_01,7,motorcycle,HIGH,inbound,zone_entry,0.8702,2026-05-02 15:40:48.968
2,03f94e20-5f11-45ea-860a-14fd0197cf32,CAM_01,5,motorcycle,HIGH,inbound,zone_entry,0.8726,2026-05-02 15:40:48.967


In [ ]:
import requests
print("⚙️ Kích hoạt Test: Chạy Aggregation và Dự báo ML...")

print("1. Gọi POST /aggregation/compute...")
res_agg = requests.post("http://127.0.0.1:8000/aggregation/compute")
if res_agg.status_code == 200:
    print("   ✅ Gom nhóm thành công! Đã lưu vào collection traffic_aggregation.")
else:
    print(f"   ⚠️ Lỗi gom nhóm: {res_agg.text}")

print("2. Gọi GET /predict-next...")
res_pred = requests.get("http://127.0.0.1:8000/predict-next")
if res_pred.status_code == 200:
    print("   ✅ Dự báo thành công! Kết quả trả về từ XGBoost:")
    print(res_pred.json())
else:
    print(f"   ⚠️ Lỗi dự báo: {res_pred.text}")

In [8]:
print("🧹 Đang dọn dẹp hệ thống...")

if 'backend_process' in locals():
    backend_process.terminate()
    backend_process.wait()
    print("✅ Đã tắt Backend an toàn. Cổng 8000 đã được giải phóng.")
else:
    print("⚠️ Không tìm thấy tiến trình Backend để tắt.")

🧹 Đang dọn dẹp hệ thống...
⚠️ Không tìm thấy tiến trình Backend để tắt.
